<a href="https://colab.research.google.com/github/wunann03/AmplifAi-Bootcamp-Basics-of-Neural-Networks/blob/Forward-Propagation-Exercise/Regression_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch

In [2]:
!curl -L -o house_prediction.zip\
   https://www.kaggle.com/api/v1/datasets/download/muhamedumarjamil/house-price-prediction-dataset

!unzip  house_prediction.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  292k  100  292k    0     0   298k      0 --:--:-- --:--:-- --:--:-- 1025k
Archive:  house_prediction.zip
  inflating: house_prices_dataset.csv  


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
# --- 1. Define the Neural Network Model ---
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(SimpleNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size1),
            nn.ReLU(),
            nn.Linear(hidden_size1, hidden_size2),
            nn.ReLU(),
            nn.Linear(hidden_size2, output_size)
        )

    def forward(self, x):
        return self.network(x)

In [5]:
# --- 2. Load and Prepare the Data ---
try:
    df = pd.read_csv('/content/house_prices_dataset.csv')
    print("Dataset loaded successfully from 'house_prices.csv'.")
except FileNotFoundError:
    print("Error: 'house_prices.csv' not found.")
    print("Please make sure the file is in the same directory as this script.")
    exit()

# Define the features (X) and the target variable (y)
features = ['square_feet', 'num_rooms', 'age', 'distance_to_city(km)']
X = df[features].values  # Features must be a 2D array
y = df[['price']].values   # Target must also be a 2D array

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the data using StandardScaler from scikit-learn
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

# Transform the test data using the *same* scalers
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

Dataset loaded successfully from 'house_prices.csv'.


In [8]:
# --- 3. Instantiate Model, Loss Function, and Optimizer ---
input_size = len(features)  # Now 4 for the house features
hidden_size1 = 64 #can be tuned
hidden_size2 = 32
output_size = 1

model = SimpleNN(input_size, hidden_size1, hidden_size2, output_size)
criterion = nn.MSELoss()  # Mean Squared Error is standard for regression
optimizer = optim.Adam(model.parameters(), lr=0.001) #lr=learning rate

In [9]:
# --- 4. Train the Neural Network ---
num_epochs = 50000
print("\nTraining the model...")
for epoch in range(num_epochs):
    # Forward pass
    model.train()  # Set the model to training mode
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 2000 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete!")


Training the model...
Epoch [2000/50000], Loss: 0.0404
Epoch [4000/50000], Loss: 0.0387
Epoch [6000/50000], Loss: 0.0377
Epoch [8000/50000], Loss: 0.0370
Epoch [10000/50000], Loss: 0.0366
Epoch [12000/50000], Loss: 0.0363
Epoch [14000/50000], Loss: 0.0361
Epoch [16000/50000], Loss: 0.0359
Epoch [18000/50000], Loss: 0.0357
Epoch [20000/50000], Loss: 0.0356
Epoch [22000/50000], Loss: 0.0353
Epoch [24000/50000], Loss: 0.0352
Epoch [26000/50000], Loss: 0.0352
Epoch [28000/50000], Loss: 0.0350
Epoch [30000/50000], Loss: 0.0349
Epoch [32000/50000], Loss: 0.0348
Epoch [34000/50000], Loss: 0.0347
Epoch [36000/50000], Loss: 0.0346
Epoch [38000/50000], Loss: 0.0347
Epoch [40000/50000], Loss: 0.0344
Epoch [42000/50000], Loss: 0.0343
Epoch [44000/50000], Loss: 0.0345
Epoch [46000/50000], Loss: 0.0344
Epoch [48000/50000], Loss: 0.0341
Epoch [50000/50000], Loss: 0.0340
Training complete!


In [10]:
# --- 5. Evaluate the Model on the Test Set ---
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # No need to compute gradients during evaluation
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)

    # Inverse transform the scaled predictions and actual values
    test_predictions = scaler_y.inverse_transform(test_outputs.numpy())
    test_actual = scaler_y.inverse_transform(y_test_tensor.numpy())

    # Calculate R-squared
    r2 = r2_score(test_actual, test_predictions)

    print("\n--- Model Evaluation ---")
    print(f'Test Loss (Mean Squared Error): {test_loss.item():.4f}')
    print(f'R-squared (R²): {r2:.4f}')


--- Model Evaluation ---
Test Loss (Mean Squared Error): 0.0478
R-squared (R²): 0.9522


In [13]:
# --- 6. Make a Prediction for a new value ---
# Let's predict the price for a new house with specific values
new_house_data = np.array([[1800, 4, 10, 7.0]])
print("\nPredicting price for a new house with the following features:")
print(f"Square Feet: {new_house_data[0][0]}, Rooms: {new_house_data[0][1]}, Age: {new_house_data[0][2]}, Dist. to City: {new_house_data[0][3]}")

# First, we need to normalize the new input values using the same scaler
new_house_scaled = scaler_X.transform(new_house_data)
new_house_tensor = torch.tensor(new_house_scaled, dtype=torch.float32)

# Make the prediction with the scaled input
model.eval()  # Set the model to evaluation mode
with torch.no_grad():  # No need to compute gradients during prediction
    predicted_price_scaled = model(new_house_tensor).item()

# Inverse transform the prediction back to the original scale
predicted_price = scaler_y.inverse_transform([[predicted_price_scaled]])[0][0]

print(f'\nPredicted House Price: ${predicted_price:,.2f}')


Predicting price for a new house with the following features:
Square Feet: 1800.0, Rooms: 4.0, Age: 10.0, Dist. to City: 7.0

Predicted House Price: $291,915.79
